# Exploratory Data Analysis (EDA)
This notebook covers the main EDA for the Nedbank Transaction Volume Forecasting challenge.
We will explore:
1. **Distribution of the Target Variable** (`next_3m_txn_count`)
2. **Seasonality and Trends** in historical transactions
3. **Missingness and Demographics Insights**

In [ ]:
import pandas as pd
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Target Distribution
Let's look at `next_3m_txn_count` in `Train.csv`.

In [ ]:
train = pd.read_csv('../data/inputs/Train.csv')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(train['next_3m_txn_count'], bins=50, ax=ax1, kde=True)
ax1.set_title('Original Target Distribution')

sns.histplot(np.log1p(train['next_3m_txn_count']), bins=50, ax=ax2, kde=True, color='orange')
ax2.set_title('Log1p Transformed Target Distribution')

plt.tight_layout()
plt.show()

**Observation:** The target is highly skewed (long tail). Log1p transform makes it much more normal, which is why optimizing RMSE on `log1p(y)` naturally minimizes RMSLE.

## 2. Seasonality and Transaction Trends
Let's sample the 18M transaction dataset and visualize monthly transaction volumes.

In [ ]:
# Load only a few columns to save memory, or use Polars
transactions = pl.scan_parquet('../data/inputs/transactions_features.parquet')

# Extract month-year and aggregate
monthly_counts = transactions.select([
    pl.col("TransactionDate").dt.truncate("1mo").alias("month"),
    pl.lit(1).alias("count")
]).group_by("month").agg(pl.len().alias("total_txns")).sort("month").collect()

monthly_counts_pd = monthly_counts.to_pandas()

plt.figure(figsize=(14, 6))
plt.plot(monthly_counts_pd['month'], monthly_counts_pd['total_txns'], marker='o', linestyle='-')
plt.title('Total Transaction Count over Time (Seasonality)')
plt.xlabel('Date')
plt.ylabel('Number of Transactions')
plt.xticks(rotation=45)
plt.grid(True)
plt.show()

**Observation:** Transaction volume may show spikes near year-end (November/December/January), which corresponds precisely to our prediction window. The pipeline's time-based features will be critical here.

## 3. Demographics and Missingness

In [ ]:
demographics = pd.read_parquet('../data/inputs/demographics_clean.parquet')

missingness = demographics.isnull().mean() * 100
missingness = missingness[missingness > 0].sort_values(ascending=False)

print("Missing percentages (>0%):\n", missingness)

plt.figure(figsize=(10, 5))
sns.barplot(x=missingness.values, y=missingness.index)
plt.title('Missing Values by Column (%)')
plt.xlabel('Percentage Missing')
plt.show()